In [ ]:
import gc
import io
import os
import random
import re
import urllib.request
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")

RUNTIME_ERROR = ""
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
except Exception as exc:
    torch = None
    AutoModelForCausalLM = None
    AutoTokenizer = None
    RUNTIME_ERROR = f"{type(exc).__name__}: {exc}"

HF_TOKEN = ""

REQUIRED_PACKAGES = ("numpy", "pandas", "torch", "transformers")

MODEL_LOCATIONS = {
    "Phi-3.5-mini": "",
    "LLaMA-3.2-1B": "",
    "DeepSeek-1.3B": "",
    "Gemma-2B-it": "",
}

MODEL_REVISIONS = {
    "Phi-3.5-mini": "",
    "LLaMA-3.2-1B": "",
    "DeepSeek-1.3B": "",
    "Gemma-2B-it": "",
}

JENA_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"

DATA_LOCATIONS = {
    "JenaClimate": "",
    "AzureLLMTrace": "",
    "ETTh1": "",
    "ETTm1": "",
    "Weather": "",
    "ECL": "",
}

DATA_TARGETS = {
    "JenaClimate": "T (degC)",
    "AzureLLMTrace": "ContextTokens",
    "ETTh1": "OT",
    "ETTm1": "OT",
    "Weather": "OT",
    "ECL": "",
}

EXPECTED_ROWS = {
    "JenaClimate": 70041,
    "AzureLLMTrace": 200000,
    "ETTh1": 17420,
    "ETTm1": 69680,
    "Weather": 52695,
    "ECL": 26304,
}

ONE_STEP_DATASETS = ("JenaClimate", "AzureLLMTrace")
MULTI_HORIZON_DATASETS = ("JenaClimate", "AzureLLMTrace", "ETTh1", "ETTm1", "Weather", "ECL")
HORIZONS = (96, 192, 336, 720)
SEED = 42
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
INITIAL_TEST_ORIGINS = 256
ONE_STEP_WINDOWS = 256
MULTI_HORIZON_WINDOWS = 8
ONE_STEP_INPUT_LENGTH = 24
MULTI_HORIZON_INPUT_LENGTH = 96
BLOCK_SIZE = 168
MAX_NEW_TOKENS = 4096
MAX_PROMPT_TOKENS = 8192
GENERATION_RETRIES = 2
RANGE_SIGMA = 4.0
RANGE_MARGIN = 1.0
REPORTED_TEMPERATURE = 0.12
REPORTED_TOP_P = 0.90

_DATA_CACHE = {}
_NUMBER_PATTERN = re.compile(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?")


def set_seed(seed=SEED):
    random.seed(int(seed))
    np.random.seed(int(seed))
    if torch is not None:
        torch.manual_seed(int(seed))
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(int(seed))
        try:
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
        except Exception:
            pass


def configured_models():
    return [name for name, location in MODEL_LOCATIONS.items() if str(location).strip()]


def safe_error(exc):
    text = f"{type(exc).__name__}: {exc}"
    private_values = [HF_TOKEN]
    private_values.extend(str(value) for value in MODEL_LOCATIONS.values())
    private_values.extend(str(value) for value in DATA_LOCATIONS.values())
    for value in private_values:
        if value:
            text = text.replace(value, "<configured_value>")
    return text[:1000]


def infer_date_column(frame):
    accepted = {"date", "ds", "time", "timestamp", "datetime", "date time"}
    for column in frame.columns:
        if str(column).lower() in accepted:
            return column
    return None


def clean_numeric_frame(frame):
    frame = frame.copy()
    date_column = infer_date_column(frame)
    if date_column is not None:
        frame[date_column] = pd.to_datetime(frame[date_column], errors="coerce")
        frame = frame.sort_values(date_column, kind="stable")
    for column in frame.columns:
        if column != date_column:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
    numeric_columns = [column for column in frame.columns if pd.api.types.is_numeric_dtype(frame[column])]
    kept_columns = ([date_column] if date_column is not None else []) + numeric_columns
    return frame[kept_columns].replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)


def read_jena_frame():
    location = str(DATA_LOCATIONS["JenaClimate"]).strip()
    if location:
        frame = pd.read_csv(Path(location))
    else:
        with urllib.request.urlopen(JENA_URL, timeout=180) as response:
            payload = response.read()
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            member = next(name for name in archive.namelist() if name.lower().endswith(".csv"))
            with archive.open(member) as stream:
                frame = pd.read_csv(stream)
    date_column = infer_date_column(frame)
    if date_column is not None and len(frame) > EXPECTED_ROWS["JenaClimate"]:
        frame[date_column] = pd.to_datetime(frame[date_column], format="%d.%m.%Y %H:%M:%S", errors="coerce")
        frame = frame.dropna(subset=[date_column]).set_index(date_column).resample("1h").mean(numeric_only=True).reset_index()
    return frame


def read_local_frame(name):
    location = str(DATA_LOCATIONS[name]).strip()
    if not location:
        return None
    path = Path(location)
    if not path.is_file():
        raise FileNotFoundError(f"{name} data file was not found")
    frame = pd.read_csv(path)
    if name == "Weather":
        date_column = infer_date_column(frame)
        if date_column is not None:
            frame[date_column] = pd.to_datetime(frame[date_column], errors="coerce")
            frame = frame.sort_values(date_column, kind="stable").drop_duplicates(subset=[date_column], keep="first")
    return frame


def resolve_target(name, frame):
    frame = clean_numeric_frame(frame)
    numeric_columns = [column for column in frame.columns if pd.api.types.is_numeric_dtype(frame[column])]
    requested = DATA_TARGETS[name]
    aliases = {
        "JenaClimate": (requested, "OT"),
        "AzureLLMTrace": (requested, "context_tokens", "tokens", "OT"),
        "ETTh1": (requested,),
        "ETTm1": (requested,),
        "Weather": (requested, "T (degC)", "temperature"),
    }
    if name == "ECL":
        if requested and requested in numeric_columns:
            values = frame[requested]
        else:
            if not numeric_columns:
                raise ValueError("ECL has no numeric client streams")
            values = frame[numeric_columns].mean(axis=1)
    else:
        target = next((column for column in aliases[name] if column in numeric_columns), None)
        if target is None:
            lowered = {str(column).lower(): column for column in numeric_columns}
            target = next((lowered[str(column).lower()] for column in aliases[name] if str(column).lower() in lowered), None)
        if target is None:
            raise ValueError(f"{name} target column was not found")
        values = frame[target]
    result = pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna().to_numpy(np.float64)
    if name == "AzureLLMTrace":
        result = result[-EXPECTED_ROWS[name]:]
    if len(result) != EXPECTED_ROWS[name]:
        raise ValueError(f"{name} has {len(result)} rows; expected {EXPECTED_ROWS[name]}")
    return result


def load_dataset(name):
    if name in _DATA_CACHE:
        return _DATA_CACHE[name]
    if name == "JenaClimate":
        frame = read_jena_frame()
    else:
        frame = read_local_frame(name)
        if frame is None:
            return None
    values = resolve_target(name, frame)
    _DATA_CACHE[name] = values
    return values


def evenly_select(values, count):
    values = np.asarray(values, dtype=int)
    if len(values) <= int(count):
        return values
    indices = np.linspace(0, len(values) - 1, int(count)).astype(int)
    return values[indices]


def prepare_windows(values, input_length, horizon, measured_windows):
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    total = len(values)
    train_end = int(total * TRAIN_RATIO)
    validation_end = int(total * (TRAIN_RATIO + VALIDATION_RATIO))
    training_values = values[:train_end]
    mean = float(np.mean(training_values))
    scale = float(np.std(training_values, ddof=0))
    if not np.isfinite(scale) or scale <= 0:
        scale = 1.0
    scaled = ((values - mean) / scale).astype(np.float32)
    valid_origins = np.arange(max(int(input_length), validation_end), max(int(input_length), total - int(horizon) + 1), dtype=int)
    origin_pool = evenly_select(valid_origins, INITIAL_TEST_ORIGINS)
    origins = origin_pool[:int(measured_windows)]
    if len(origins) != int(measured_windows):
        raise ValueError(f"Only {len(origins)} valid test origins are available")
    histories = np.stack([scaled[origin - int(input_length):origin] for origin in origins]).astype(np.float32)
    targets = np.stack([scaled[origin:origin + int(horizon)] for origin in origins]).astype(np.float32)
    return {
        "mean": mean,
        "scale": scale,
        "train_scaled": scaled[:train_end],
        "origins": origins,
        "histories": histories,
        "targets": targets,
    }


class STM:
    def __init__(self, k=5, t_max=96, period_tolerance=1, period_threshold=0.90, period_near_best=0.05, period_min_cycles=3, period_prominence_threshold=4.0):
        self.k = int(k)
        self.t_max = int(t_max)
        self.period_tolerance = int(period_tolerance)
        self.period_threshold = float(period_threshold)
        self.period_near_best = float(period_near_best)
        self.period_min_cycles = int(period_min_cycles)
        self.period_prominence_threshold = float(period_prominence_threshold)
        self.symbols = ["VL", "L", "M", "H", "VH"]
        self.edges = None
        if self.k != 5:
            raise ValueError("The public protocol fixes k at 5")

    def fit(self, training_values):
        values = np.asarray(training_values, dtype=float).reshape(-1)
        values = values[np.isfinite(values)]
        if not len(values):
            raise ValueError("STM requires finite training values")
        low = float(np.min(values))
        high = float(np.max(values))
        if high <= low:
            high = low + 1e-6
        edges = np.linspace(low, high, self.k + 1)
        edges[0] = -np.inf
        edges[-1] = np.inf
        self.edges = edges
        return self

    def transform_codes(self, values):
        if self.edges is None:
            raise RuntimeError("STM must be fitted on the training partition")
        return np.clip(np.digitize(np.asarray(values, dtype=float), self.edges[1:-1]), 0, self.k - 1).astype(int)

    def detect_period(self, deltas):
        deltas = np.asarray(deltas, dtype=int).reshape(-1)
        length = len(deltas)
        if length < 2 * self.period_min_cycles:
            return {"period": None, "score": 0.0, "prominence": 0.0}
        maximum_period = min(self.t_max, length // max(2, self.period_min_cycles))
        scores = {
            period: float(np.mean(np.abs(deltas[:-period] - deltas[period:]) <= self.period_tolerance))
            for period in range(2, maximum_period + 1)
        }
        if not scores:
            return {"period": None, "score": 0.0, "prominence": 0.0}
        all_scores = np.asarray(list(scores.values()), dtype=float)
        prominence = {}
        for period, score in scores.items():
            background = np.asarray([
                value for other, value in scores.items()
                if other % period != 0 and period % other != 0
            ], dtype=float)
            if len(background) < 5:
                background = all_scores
            median = float(np.median(background))
            mad = float(np.median(np.abs(background - median)) + 1e-6)
            prominence[period] = (score - median) / mad
        best = max(scores.values())
        accepted = [
            period for period, score in scores.items()
            if score >= self.period_threshold
            and score >= best - self.period_near_best
            and prominence[period] >= self.period_prominence_threshold
        ]
        period = min(accepted) if accepted else None
        return {
            "period": period,
            "score": float(scores[period] if period is not None else best),
            "prominence": float(prominence[period] if period is not None else max(prominence.values())),
        }

    def compute(self, values):
        codes = self.transform_codes(values)
        deltas = np.diff(codes, prepend=codes[0]).astype(int)
        period_information = self.detect_period(deltas)
        return {
            "codes": codes,
            "symbols": [self.symbols[code] for code in codes],
            "deltas": deltas,
            "period": period_information["period"],
            "period_score": period_information["score"],
        }


def format_numbers(values, precision=4):
    return ", ".join(f"{float(value):.{int(precision)}f}" for value in np.asarray(values).reshape(-1))


def make_one_step_prompt(sequence, variant, stm, dataset_name):
    numeric = format_numbers(sequence)
    if dataset_name == "JenaClimate":
        task = f"The normalized temperature readings for the past 24 hours are: {numeric}."
        question = "What is the next normalized temperature reading?"
    else:
        task = f"Given the following sequence of normalized inference traffic values: {numeric}."
        question = "Predict the next normalized traffic value based on the pattern and trend."
    symbolic = ""
    if variant == "full_stm":
        symbolic = " STM symbolic pattern: " + " ".join(stm.compute(sequence)["symbols"]) + "."
    elif variant != "raw":
        raise ValueError("variant must be raw or full_stm")
    return task + symbolic + " " + question + " Return exactly one number and no explanation."


def make_representation_prompt(sequence, variant, stm):
    numeric = format_numbers(sequence)
    base = "Encode the observed normalized univariate time series for forecasting. " + f"Numeric sequence: {numeric}. "
    if variant == "raw":
        return base + "Use the numeric level, trend, and recent dynamics."
    if variant != "full_stm":
        raise ValueError("variant must be raw or full_stm")
    representation = stm.compute(sequence)
    pieces = ["Symbolic levels: " + " ".join(representation["symbols"]) + "."]
    pieces.append(
        "Mean absolute symbolic transition: "
        + f"{np.mean(np.abs(representation['deltas'])):.4f}; "
        + "net symbolic direction: "
        + f"{int(np.sum(representation['deltas']))}."
    )
    if representation["period"] is None:
        pieces.append("No accepted repeating interval; best reliability: " + f"{representation['period_score']:.4f}.")
    else:
        pieces.append(
            "Detected repeating interval: "
            + f"{representation['period']}; reliability: "
            + f"{representation['period_score']:.4f}."
        )
    return base + "External symbolic prompt guidance: " + " ".join(pieces)


def make_multi_horizon_prompt(sequence, requested_values, variant, stm):
    representation = make_representation_prompt(sequence, variant, stm)
    return (
        "You are forecasting a normalized univariate time series. "
        + representation
        + f" Forecast exactly the next {int(requested_values)} consecutive normalized values. "
        + "Return only one bracketed comma-separated numeric list, with no labels, indices, units, or explanation."
    )


def parse_generated_numbers(text, requested_values):
    groups = re.findall(r"[\[\(]([^\]\)]{1,20000})[\]\)]", str(text), flags=re.S)
    candidates = groups + [str(text)]
    best = []
    for candidate in candidates:
        parsed = []
        for token in _NUMBER_PATTERN.findall(candidate.replace(",", " ")):
            try:
                parsed.append(float(token))
            except Exception:
                pass
        if len(parsed) > len(best):
            best = parsed
    output = np.full(int(requested_values), np.nan, dtype=float)
    count = min(len(best), int(requested_values))
    if count:
        output[:count] = np.asarray(best[:count], dtype=float)
    return output, int(count)


def symmetric_history_guard(prediction, history):
    prediction = np.asarray(prediction, dtype=float).copy()
    history = np.asarray(history, dtype=float).reshape(-1)
    last_value = float(history[-1])
    prediction = np.where(np.isfinite(prediction), prediction, last_value)
    mean = float(np.mean(history))
    standard_deviation = float(np.std(history, ddof=0) + 1e-6)
    lower = min(float(np.min(history)) - RANGE_MARGIN * standard_deviation, mean - RANGE_SIGMA * standard_deviation)
    upper = max(float(np.max(history)) + RANGE_MARGIN * standard_deviation, mean + RANGE_SIGMA * standard_deviation)
    return np.clip(prediction, lower, upper)


def load_frozen_model(model_name):
    if RUNTIME_ERROR or torch is None or AutoTokenizer is None or AutoModelForCausalLM is None:
        raise RuntimeError("PyTorch and Transformers are required")
    location = str(MODEL_LOCATIONS[model_name]).strip()
    if not location:
        raise ValueError(f"Set MODEL_LOCATIONS for {model_name}")
    revision = str(MODEL_REVISIONS[model_name]).strip() or None
    common = {"token": HF_TOKEN or None, "revision": revision}
    common = {key: value for key, value in common.items() if value is not None}
    tokenizer = None
    tokenizer_error = None
    for trust_remote_code in (False, True):
        try:
            tokenizer = AutoTokenizer.from_pretrained(location, trust_remote_code=trust_remote_code, **common)
            break
        except Exception as exc:
            tokenizer_error = exc
    if tokenizer is None:
        raise RuntimeError(f"Tokenizer loading failed: {tokenizer_error}")
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token
    if tokenizer.pad_token_id is None:
        raise RuntimeError("Tokenizer has no usable pad token")
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = None
    model_error = None
    for trust_remote_code in (False, True):
        base = {"trust_remote_code": trust_remote_code, "low_cpu_mem_usage": True, **common}
        if torch.cuda.is_available():
            base["device_map"] = {"": 0}
        for dtype_key in ("dtype", "torch_dtype"):
            arguments = dict(base)
            arguments[dtype_key] = dtype
            try:
                model = AutoModelForCausalLM.from_pretrained(location, **arguments)
                break
            except TypeError as exc:
                model_error = exc
            except Exception as exc:
                model_error = exc
                break
        if model is not None:
            break
    if model is None:
        raise RuntimeError(f"Model loading failed: {model_error}")
    if not torch.cuda.is_available():
        model.to("cpu")
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    model._stm_no_cache = False
    if any(parameter.requires_grad for parameter in model.parameters()):
        raise RuntimeError("The backbone was not fully frozen")
    return tokenizer, model


def release_model(tokenizer, model):
    del tokenizer
    del model
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()


def is_cache_error(exc):
    message = str(exc).lower()
    keys = ("dynamiccache", "past_key_values", "cache_position", "seen_tokens", "get_max_length", "sizes of tensors must match", "sliding_window")
    return any(key in message for key in keys)


def generate_completion(tokenizer, model, prompt, maximum_new_tokens):
    messages = [{"role": "user", "content": prompt}]
    try:
        rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        rendered = prompt
    encoded = tokenizer(rendered, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_TOKENS)
    encoded = {key: value.to(model.device) for key, value in encoded.items()}
    arguments = {
        "max_new_tokens": int(maximum_new_tokens),
        "do_sample": False,
        "num_return_sequences": 1,
        "pad_token_id": tokenizer.pad_token_id,
        "use_cache": not bool(getattr(model, "_stm_no_cache", False)),
    }
    try:
        with torch.inference_mode():
            output = model.generate(**encoded, **arguments)
    except Exception as exc:
        if arguments["use_cache"] and is_cache_error(exc):
            model._stm_no_cache = True
            arguments["use_cache"] = False
            with torch.inference_mode():
                output = model.generate(**encoded, **arguments)
        else:
            raise
    generated = output[0, encoded["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


def forecast_one_step(tokenizer, model, history, variant, stm, dataset_name):
    prompt = make_one_step_prompt(history, variant, stm, dataset_name)
    completion = generate_completion(tokenizer, model, prompt, 64)
    values, count = parse_generated_numbers(completion, 1)
    if not count or not np.isfinite(values[0]):
        values[0] = float(np.asarray(history).reshape(-1)[-1])
    return float(values[0]), int(count)


def forecast_block(tokenizer, model, history, count, variant, stm):
    prompt = make_multi_horizon_prompt(history, count, variant, stm)
    best = np.full(int(count), np.nan, dtype=float)
    best_count = 0
    budget = min(MAX_NEW_TOKENS, max(64, 12 * int(count) + 32))
    for attempt in range(GENERATION_RETRIES):
        current_prompt = prompt if attempt == 0 else prompt + " Your previous format was incomplete; output the complete numeric list now."
        completion = generate_completion(tokenizer, model, current_prompt, budget)
        values, parsed_count = parse_generated_numbers(completion, count)
        if parsed_count > best_count:
            best = values
            best_count = parsed_count
        if parsed_count >= int(count):
            break
    return best, int(best_count)


def forecast_multi_horizon(tokenizer, model, input_sequence, horizon, variant, stm):
    context = list(np.asarray(input_sequence, dtype=float).reshape(-1))
    predictions = []
    parsed = 0
    requested = 0
    while len(predictions) < int(horizon):
        count = min(BLOCK_SIZE, int(horizon) - len(predictions))
        recent_history = np.asarray(context[-MULTI_HORIZON_INPUT_LENGTH:], dtype=float)
        values, parsed_count = forecast_block(tokenizer, model, recent_history, count, variant, stm)
        values = symmetric_history_guard(values, recent_history)
        predictions.extend(values.tolist())
        context.extend(values.tolist())
        parsed += min(parsed_count, count)
        requested += count
    return np.asarray(predictions[:int(horizon)], dtype=float), parsed / max(1, requested)


def inverse_scale(values, mean, scale):
    return np.asarray(values, dtype=float) * float(scale) + float(mean)


def core_self_test():
    stm = STM().fit(np.linspace(-2.0, 2.0, 101))
    codes = stm.transform_codes(np.array([-3.0, -1.0, 0.0, 1.0, 3.0]))
    periodic = stm.detect_period(np.tile(np.array([2, -1, 0, 1]), 12))
    parsed, count = parse_generated_numbers("[-1.25, 2.5e-1]", 2)
    raw = make_representation_prompt(np.linspace(-1.0, 1.0, 8), "raw", stm)
    full = make_representation_prompt(np.linspace(-1.0, 1.0, 8), "full_stm", stm)
    numeric = "Numeric sequence: " + format_numbers(np.linspace(-1.0, 1.0, 8)) + ". "
    passed = bool(
        codes.min() == 0
        and codes.max() == 4
        and periodic["period"] == 4
        and count == 2
        and np.allclose(parsed, [-1.25, 0.25])
        and numeric in raw
        and numeric in full
    )
    return "passed" if passed else "failed"


set_seed()
print("result:", {
    "status": "ready" if not RUNTIME_ERROR else "runtime_configuration_required",
    "runtime": "ready" if not RUNTIME_ERROR else RUNTIME_ERROR,
    "required_packages": REQUIRED_PACKAGES,
    "required_models": list(MODEL_LOCATIONS),
    "configured_models": configured_models(),
    "jena_source": JENA_URL,
    "local_datasets_to_add": [name for name in DATA_LOCATIONS if name != "JenaClimate"],
    "core_self_test": core_self_test(),
})

In [ ]:
def run_one_step_experiment():
    models = configured_models()
    if not models:
        return {
            "status": "configuration_required",
            "required_models": list(MODEL_LOCATIONS),
            "required_datasets": list(ONE_STEP_DATASETS),
        }
    bundles = {}
    missing_datasets = []
    errors = []
    for dataset_name in ONE_STEP_DATASETS:
        try:
            values = load_dataset(dataset_name)
            if values is None:
                missing_datasets.append(dataset_name)
                continue
            bundles[dataset_name] = prepare_windows(values, ONE_STEP_INPUT_LENGTH, 1, ONE_STEP_WINDOWS)
        except Exception as exc:
            errors.append({"dataset": dataset_name, "error": safe_error(exc)})
    if not bundles:
        return {
            "status": "configuration_required",
            "required_models": list(MODEL_LOCATIONS),
            "required_datasets": list(ONE_STEP_DATASETS),
            "missing_datasets": missing_datasets,
            "errors": errors,
        }
    results = []
    for model_name in models:
        tokenizer = None
        model = None
        try:
            tokenizer, model = load_frozen_model(model_name)
            for dataset_name, bundle in bundles.items():
                stm = STM().fit(bundle["train_scaled"])
                for variant in ("raw", "full_stm"):
                    predictions = []
                    parsed = []
                    for history in bundle["histories"]:
                        prediction, parsed_count = forecast_one_step(tokenizer, model, history, variant, stm, dataset_name)
                        predictions.append(prediction)
                        parsed.append(parsed_count)
                    predictions = np.asarray(predictions, dtype=float)
                    targets = bundle["targets"][:, 0]
                    predictions_original = inverse_scale(predictions, bundle["mean"], bundle["scale"])
                    targets_original = inverse_scale(targets, bundle["mean"], bundle["scale"])
                    results.append({
                        "dataset": dataset_name,
                        "model": model_name,
                        "variant": variant,
                        "test_origins": int(len(targets)),
                        "MAE": round(float(np.mean(np.abs(targets_original - predictions_original))), 8),
                        "MSE": round(float(np.mean((targets_original - predictions_original) ** 2)), 8),
                        "parsed_fraction": round(float(np.mean(parsed)), 8),
                    })
        except Exception as exc:
            errors.append({"model": model_name, "error": safe_error(exc)})
        finally:
            if tokenizer is not None and model is not None:
                release_model(tokenizer, model)
                tokenizer = None
                model = None
    return {
        "status": "ok" if results and not errors and not missing_datasets else "partial" if results else "failed",
        "protocol": {
            "input_length": ONE_STEP_INPUT_LENGTH,
            "horizon": 1,
            "test_origins": ONE_STEP_WINDOWS,
            "do_sample": False,
            "temperature_reported": REPORTED_TEMPERATURE,
            "top_p_reported": REPORTED_TOP_P,
        },
        "results": results,
        "missing_datasets": missing_datasets,
        "errors": errors,
    }


print("result:", run_one_step_experiment())

In [ ]:
def run_multi_horizon_experiment():
    models = configured_models()
    if not models:
        return {
            "status": "configuration_required",
            "required_models": list(MODEL_LOCATIONS),
            "required_datasets": list(MULTI_HORIZON_DATASETS),
        }
    bundles = {}
    missing_datasets = []
    errors = []
    for dataset_name in MULTI_HORIZON_DATASETS:
        try:
            values = load_dataset(dataset_name)
            if values is None:
                missing_datasets.append(dataset_name)
                continue
            for horizon in HORIZONS:
                bundles[(dataset_name, int(horizon))] = prepare_windows(
                    values,
                    MULTI_HORIZON_INPUT_LENGTH,
                    int(horizon),
                    MULTI_HORIZON_WINDOWS,
                )
        except Exception as exc:
            errors.append({"dataset": dataset_name, "error": safe_error(exc)})
    if not bundles:
        return {
            "status": "configuration_required",
            "required_models": list(MODEL_LOCATIONS),
            "required_datasets": list(MULTI_HORIZON_DATASETS),
            "missing_datasets": missing_datasets,
            "errors": errors,
        }
    results = []
    for model_name in models:
        tokenizer = None
        model = None
        try:
            tokenizer, model = load_frozen_model(model_name)
            for (dataset_name, horizon), bundle in bundles.items():
                stm = STM().fit(bundle["train_scaled"])
                for variant in ("raw", "full_stm"):
                    window_nmae = []
                    parsed_fractions = []
                    for history, target in zip(bundle["histories"], bundle["targets"]):
                        prediction, parsed_fraction = forecast_multi_horizon(tokenizer, model, history, horizon, variant, stm)
                        window_nmae.append(float(np.mean(np.abs(np.asarray(target, dtype=float) - prediction))))
                        parsed_fractions.append(float(parsed_fraction))
                    results.append({
                        "dataset": dataset_name,
                        "horizon": int(horizon),
                        "model": model_name,
                        "variant": variant,
                        "test_origins": int(len(window_nmae)),
                        "NMAE_mean": round(float(np.mean(window_nmae)), 8),
                        "NMAE_std": round(float(np.std(window_nmae, ddof=1)), 8),
                        "parsed_fraction": round(float(np.mean(parsed_fractions)), 8),
                    })
        except Exception as exc:
            errors.append({"model": model_name, "error": safe_error(exc)})
        finally:
            if tokenizer is not None and model is not None:
                release_model(tokenizer, model)
                tokenizer = None
                model = None
    return {
        "status": "ok" if results and not errors and not missing_datasets else "partial" if results else "failed",
        "protocol": {
            "input_length": MULTI_HORIZON_INPUT_LENGTH,
            "horizons": HORIZONS,
            "test_origins": MULTI_HORIZON_WINDOWS,
            "initial_origin_pool": INITIAL_TEST_ORIGINS,
            "block_size": BLOCK_SIZE,
            "do_sample": False,
            "temperature_reported": REPORTED_TEMPERATURE,
            "top_p_reported": REPORTED_TOP_P,
        },
        "results": results,
        "missing_datasets": missing_datasets,
        "errors": errors,
    }


print("result:", run_multi_horizon_experiment())